In [1]:
import boto3
import os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from io import BytesIO

from ilipy import Session
from ilipy.database import DistanceCorrelation
from ilipy import TrackIndex, OdometerTicks

inspection_id = "0AS0KUQOE45"
num_tracks = 22
environment = "prod"
model_detection_threshold = 0.99


Loading stub library for: libnppial.so.11
Loading stub library for: libnppig.so.11
Loading stub library for: libnppist.so.11
Loading stub library for: libnppc.so.11


In [2]:
def download_parquet_from_s3(bucket_name, s3_key, local_path=None):
    """
    Download a parquet file from S3 and optionally save locally or return as DataFrame
    
    Args:
        bucket_name: S3 bucket name
        s3_key: S3 key (path) for the file
        local_path: Optional local file path to save the file. If None, returns DataFrame
        
    Returns:
        DataFrame if local_path is None, otherwise saves file locally and returns file path
    """
    s3_client = boto3.client('s3')
    
    try:
        # Download file from S3
        response = s3_client.get_object(Bucket=bucket_name, Key=s3_key)
        
        if local_path:
            # Save to local file
            with open(local_path, 'wb') as f:
                f.write(response['Body'].read())
            print(f"File downloaded and saved to: {local_path}")
            return local_path
        else:
            # Return as DataFrame
            parquet_buffer = BytesIO(response['Body'].read())
            df = pd.read_parquet(parquet_buffer)
            print(f"Downloaded DataFrame with {len(df)} rows from s3://{bucket_name}/{s3_key}")
            return df
            
    except Exception as e:
        print(f"Error downloading file: {e}")
        return None


In [3]:
def add_view_distances(df, dist_corr, track_col='track_id', start_odo_col='start_odometer_tick', end_odo_col='end_odometer_tick'):
    """
    Add view_distance_start and view_distance_stop columns to DataFrame
    
    Args:
        df: DataFrame with odometer tick columns
        dist_corr: DistanceCorrelation object from ilipy
        track_col: column name containing track IDs
        start_odo_col: column name containing start odometer ticks
        end_odo_col: column name containing end odometer ticks
    
    Returns:
        DataFrame with added view_distance_start and view_distance_stop columns
    """
    if df is None or len(df) == 0:
        return df
    
    df = df.copy()
    
    # Initialize new columns
    df['view_distance_start'] = np.nan
    df['view_distance_stop'] = np.nan
    
    # Process each row
    for idx, row in df.iterrows():
        try:
            track_id = int(row[track_col])
            start_odo = int(row[start_odo_col])
            end_odo = int(row[end_odo_col])
            
            # Get view distances
            start_vd = dist_corr.get_view_distance_from_odometer_ticks(
                TrackIndex(track_id), 
                OdometerTicks(start_odo)
            ).value
            
            end_vd = dist_corr.get_view_distance_from_odometer_ticks(
                TrackIndex(track_id), 
                OdometerTicks(end_odo)
            ).value
            df.loc[idx, 'view_distance_start'] = start_vd
            df.loc[idx, 'view_distance_stop'] = end_vd
            
        except Exception as e:
            print(f"Error processing row {idx}: {e}")
            continue
    return df

In [4]:
from ilipy import ViewDistance, PipeDistance

def add_pipe_distances(df, dist_corr, 
                       view_distance_start_col='view_distance_start', 
                       view_distance_stop_col='view_distance_stop'):
    """
    Add pipe_distance_start and pipe_distance_stop columns to a DataFrame
    by converting view distances to pipe distances using anchor-based correlation.
    
    Args:
        df: DataFrame with view distance columns
        dist_corr: DistanceCorrelation object from ilipy
        view_distance_start_col: column name for start view distance
        view_distance_stop_col: column name for stop view distance
    
    Returns:
        DataFrame with added pipe_distance_start and pipe_distance_stop columns
    """
    if df is None or len(df) == 0:
        return df
    
    df = df.copy()
    df['pipe_distance_start'] = np.nan
    df['pipe_distance_stop'] = np.nan
    
    for idx, row in df.iterrows():
        try:
            vd_start = row[view_distance_start_col]
            vd_stop = row[view_distance_stop_col]
            
            if pd.notna(vd_start):
                pd_start = dist_corr.get_pipe_distance_from_view_distance(
                    ViewDistance(float(vd_start))
                ).value
                df.loc[idx, 'pipe_distance_start'] = pd_start
            
            if pd.notna(vd_stop):
                pd_stop = dist_corr.get_pipe_distance_from_view_distance(
                    ViewDistance(float(vd_stop))
                ).value
                df.loc[idx, 'pipe_distance_stop'] = pd_stop
                
        except Exception as e:
            print(f"Error processing row {idx}: {e}")
            continue
    
    return df

In [5]:
session = Session(environment=environment)
session.set_active_inspection(inspection_id)
dist_corr = DistanceCorrelation(session)

bucket_name = "dent-dev"
num_tracks = 24
bookmarks_dfs = []
for track_idx in range(num_tracks):
    s3_key = f"dent_bookmarks/UltrasoundDent_bookmarks_{inspection_id}_track_{track_idx}.parquet"
    df = download_parquet_from_s3(bucket_name, s3_key)
    if df is not None:
        print(f"Downloaded DataFrame shape: {df.shape}")
        processed_df = add_view_distances(df, dist_corr)
        bookmarks_dfs.append(processed_df)
bookmarks_dfs = pd.concat(bookmarks_dfs, ignore_index=True)
display(bookmarks_dfs)

PySettings.cpp(46): ilipy: Build 1.2.1.14711 89815f7b8c release-ili-1.2 'Sat Apr 18 04:38:21 2026'

PySettings.cpp(47): Paths modulePath: /home/zmirikha/ilipy/lib/python3.11/site-packages/ilipy, userHomePath: /home/zmirikha, identityPath: /home/zmirikha/.aws/ilipy

PyAuthenticator.cpp(239): Attempting an IAM role login as no username and password or environment variables were provided.

CognitoAuthenticator.cpp(158): Using Cognito Credentials

CognitoAuthenticator.cpp(842): Login Successful.

CognitoAuthenticator.cpp(842): Login Successful.

CognitoAuthenticator.cpp(883): Logged in as username: zahra.mirikharaji

CognitoAuthenticator.cpp(883): Logged in as username: zahra.mirikharaji

DatabaseWebSocket.cpp(153): WebSocket connection established to https://backend.ili-prod.darkvisiontech.com/dbservice/wsbin

Versioning.cpp(1238): Database schema version=5.3.57591361FB0BAC95, code-generated schema version=5.3.57591361FB0BAC95

ObserverHost.cpp(207): [ObserverHost] Registering with dv-obs

,inspection_id,frame_range,clip_id,track_id,start_frame,end_frame,sequence_length,frame_span,num_detections,start_timer_tick,...,timer_span,start_odometer_tick,end_odometer_tick,odometer_span,avg_confidence,max_confidence,min_confidence,confidence_std,view_distance_start,view_distance_stop
0,0AS0KUQOE45,100105009_100105013,01-000-198EUYDF,0,100105009,100105013,5,5,5,1863376033,...,63,1509379286,1509379346,60,0.9903,0.9903,0.9903,0.0000,150934.720981,150934.727119
1,0AS0KUQOE45,100138954_100139033,01-000-198EUYDF,0,100138954,100139033,80,80,80,1863988282,...,1375,1509890571,1509891756,1185,0.9927,0.9934,0.9910,0.0007,150985.849473,150985.968074
2,0AS0KUQOE45,100139064_100139153,01-000-198EUYDF,0,100139064,100139153,90,90,90,1863990191,...,1564,1509892221,1509893556,1335,0.9908,0.9920,0.9900,0.0006,150986.014634,150986.148134
3,0AS0KUQOE45,100139189_100139283,01-000-198EUYDF,0,100139189,100139283,95,95,95,1863992393,...,1642,1509894096,1509895506,1410,0.9918,0.9936,0.9901,0.0012,150986.202111,150986.343168
4,0AS0KUQOE45,100141729_100141733,01-000-198EUYDF,0,100141729,100141733,5,5,5,1864036084,...,66,1509932211,1509932271,60,0.9903,0.9903,0.9903,0.0000,150990.013773,150990.019792
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30092,0AS0KUQOE45,99951679_99951683,01-021-198EVB8A,21,99951679,99951683,5,5,5,1860809757,...,67,1507212516,1507212576,60,0.9902,0.9902,0.9902,0.0000,150717.076736,150717.082713
30093,0AS0KUQOE45,99958859_99959018,01-021-198EVB8A,21,99958859,99959018,160,160,160,1860936884,...,2841,1507321406,1507323836,2430,0.9928,0.9942,0.9906,0.0009,150727.965643,150728.208832
30094,0AS0KUQOE45,99959154_99959268,01-021-198EVB8A,21,99959154,99959268,115,115,115,1860942135,...,2053,1507325906,1507327646,1740,0.9920,0.9925,0.9904,0.0005,150728.415764,150728.589824
30095,0AS0KUQOE45,99961669_99961673,01-021-198EVB8A,21,99961669,99961673,5,5,5,1860986503,...,75,1507363921,1507363981,60,0.9901,0.9901,0.9901,0.0000,150732.217364,150732.223350


CognitoAuthenticator.cpp(842): Login Successful.

CognitoAuthenticator.cpp(883): Logged in as username: zahra.mirikharaji

CognitoAuthenticator.cpp(842): Login Successful.

CognitoAuthenticator.cpp(883): Logged in as username: zahra.mirikharaji

CognitoAuthenticator.cpp(842): Login Successful.

CognitoAuthenticator.cpp(883): Logged in as username: zahra.mirikharaji

CognitoAuthenticator.cpp(842): Login Successful.

CognitoAuthenticator.cpp(883): Logged in as username: zahra.mirikharaji

CognitoAuthenticator.cpp(842): Login Successful.

CognitoAuthenticator.cpp(883): Logged in as username: zahra.mirikharaji

CognitoAuthenticator.cpp(842): Login Successful.

CognitoAuthenticator.cpp(883): Logged in as username: zahra.mirikharaji

CognitoAuthenticator.cpp(842): Login Successful.

CognitoAuthenticator.cpp(883): Logged in as username: zahra.mirikharaji

CognitoAuthenticator.cpp(842): Login Successful.

CognitoAuthenticator.cpp(883): Logged in as username: zahra.mirikharaji

CognitoAuthentic

In [6]:
display(bookmarks_dfs)

,inspection_id,frame_range,clip_id,track_id,start_frame,end_frame,sequence_length,frame_span,num_detections,start_timer_tick,...,timer_span,start_odometer_tick,end_odometer_tick,odometer_span,avg_confidence,max_confidence,min_confidence,confidence_std,view_distance_start,view_distance_stop
0,0AS0KUQOE45,100105009_100105013,01-000-198EUYDF,0,100105009,100105013,5,5,5,1863376033,...,63,1509379286,1509379346,60,0.9903,0.9903,0.9903,0.0000,150934.720981,150934.727119
1,0AS0KUQOE45,100138954_100139033,01-000-198EUYDF,0,100138954,100139033,80,80,80,1863988282,...,1375,1509890571,1509891756,1185,0.9927,0.9934,0.9910,0.0007,150985.849473,150985.968074
2,0AS0KUQOE45,100139064_100139153,01-000-198EUYDF,0,100139064,100139153,90,90,90,1863990191,...,1564,1509892221,1509893556,1335,0.9908,0.9920,0.9900,0.0006,150986.014634,150986.148134
3,0AS0KUQOE45,100139189_100139283,01-000-198EUYDF,0,100139189,100139283,95,95,95,1863992393,...,1642,1509894096,1509895506,1410,0.9918,0.9936,0.9901,0.0012,150986.202111,150986.343168
4,0AS0KUQOE45,100141729_100141733,01-000-198EUYDF,0,100141729,100141733,5,5,5,1864036084,...,66,1509932211,1509932271,60,0.9903,0.9903,0.9903,0.0000,150990.013773,150990.019792
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30092,0AS0KUQOE45,99951679_99951683,01-021-198EVB8A,21,99951679,99951683,5,5,5,1860809757,...,67,1507212516,1507212576,60,0.9902,0.9902,0.9902,0.0000,150717.076736,150717.082713
30093,0AS0KUQOE45,99958859_99959018,01-021-198EVB8A,21,99958859,99959018,160,160,160,1860936884,...,2841,1507321406,1507323836,2430,0.9928,0.9942,0.9906,0.0009,150727.965643,150728.208832
30094,0AS0KUQOE45,99959154_99959268,01-021-198EVB8A,21,99959154,99959268,115,115,115,1860942135,...,2053,1507325906,1507327646,1740,0.9920,0.9925,0.9904,0.0005,150728.415764,150728.589824
30095,0AS0KUQOE45,99961669_99961673,01-021-198EVB8A,21,99961669,99961673,5,5,5,1860986503,...,75,1507363921,1507363981,60,0.9901,0.9901,0.9901,0.0000,150732.217364,150732.223350


In [8]:
#save bookmarks_dfs
bookmarks_dfs.to_parquet(f"./insp_{inspection_id}_us_bookmarks_with_view_distances.parquet", index=False)
bookmarks_dfs = pd.read_parquet(f"./insp_{inspection_id}_us_bookmarks_with_view_distances.parquet")
display(bookmarks_dfs)

,inspection_id,frame_range,clip_id,track_id,start_frame,end_frame,sequence_length,frame_span,num_detections,start_timer_tick,...,timer_span,start_odometer_tick,end_odometer_tick,odometer_span,avg_confidence,max_confidence,min_confidence,confidence_std,view_distance_start,view_distance_stop
0,0AS0KUQOE45,100105009_100105013,01-000-198EUYDF,0,100105009,100105013,5,5,5,1863376033,...,63,1509379286,1509379346,60,0.9903,0.9903,0.9903,0.0000,150934.720981,150934.727119
1,0AS0KUQOE45,100138954_100139033,01-000-198EUYDF,0,100138954,100139033,80,80,80,1863988282,...,1375,1509890571,1509891756,1185,0.9927,0.9934,0.9910,0.0007,150985.849473,150985.968074
2,0AS0KUQOE45,100139064_100139153,01-000-198EUYDF,0,100139064,100139153,90,90,90,1863990191,...,1564,1509892221,1509893556,1335,0.9908,0.9920,0.9900,0.0006,150986.014634,150986.148134
3,0AS0KUQOE45,100139189_100139283,01-000-198EUYDF,0,100139189,100139283,95,95,95,1863992393,...,1642,1509894096,1509895506,1410,0.9918,0.9936,0.9901,0.0012,150986.202111,150986.343168
4,0AS0KUQOE45,100141729_100141733,01-000-198EUYDF,0,100141729,100141733,5,5,5,1864036084,...,66,1509932211,1509932271,60,0.9903,0.9903,0.9903,0.0000,150990.013773,150990.019792
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30092,0AS0KUQOE45,99951679_99951683,01-021-198EVB8A,21,99951679,99951683,5,5,5,1860809757,...,67,1507212516,1507212576,60,0.9902,0.9902,0.9902,0.0000,150717.076736,150717.082713
30093,0AS0KUQOE45,99958859_99959018,01-021-198EVB8A,21,99958859,99959018,160,160,160,1860936884,...,2841,1507321406,1507323836,2430,0.9928,0.9942,0.9906,0.0009,150727.965643,150728.208832
30094,0AS0KUQOE45,99959154_99959268,01-021-198EVB8A,21,99959154,99959268,115,115,115,1860942135,...,2053,1507325906,1507327646,1740,0.9920,0.9925,0.9904,0.0005,150728.415764,150728.589824
30095,0AS0KUQOE45,99961669_99961673,01-021-198EVB8A,21,99961669,99961673,5,5,5,1860986503,...,75,1507363921,1507363981,60,0.9901,0.9901,0.9901,0.0000,150732.217364,150732.223350


## Visualize all detected frames

In [9]:
# from ilipy.beamformer import BfImageType
# from ilipyutils.beamforming import Beamformer
# FF_DEFAULTS = {
#     "angle_resolution_radians": np.deg2rad(0.025),
#     "radial_resolution_mm": 0.05,
#     "radial_range_mm": [15, 33],
#     "angle_range_radians": np.deg2rad([-11, 11]).tolist(),
# }
# beamformer = Beamformer(
#     session=session,
#     inspection_id=session.active_inspection.inspection_id,
#     clip=clip_ds,
# )
# vol = beamformer.beamform_data(
#     frame_range=(iframe, iframe + 1),
#     bf_type=BfImageType.InnerSurfaceDetect,
#     calc_circles=False,
#     bf_configs=FF_DEFAULTS,
# )

import os
from pathlib import Path
import matplotlib.pyplot as plt
from ilipy.beamformer import BfImageType
from ilipyutils.beamforming import Beamformer
from ilipy import Clip, ClipTypes


def visualize_bookmark_frames(bookmarks_df, session, output_folder="./bookmark_frames"):
    """
    Save beamformed images for all frames in bookmarks dataframe.
    
    Args:
        bookmarks_df: DataFrame with bookmark data containing track_id, start_frame, end_frame
        session: ilipy Session object
        output_folder: Path to save the frame images
    """
    # Create output folder if it doesn't exist
    output_path = Path(output_folder)
    output_path.mkdir(parents=True, exist_ok=True)
    
    # Beamformer configuration
    FF_DEFAULTS = {
        "angle_resolution_radians": np.deg2rad(0.025),
        "radial_resolution_mm": 0.05,
        "radial_range_mm": [15, 33],
        "angle_range_radians": np.deg2rad([-11, 11]).tolist(),
    }
    

    print(f"Processing {len(bookmarks_df)} bookmarks...")
    clips = session.get_clips_by_type(ClipTypes.ChannelData)
    
    # Process each bookmark
    for idx, row in bookmarks_df.iterrows():
        track_id = int(row['track_id'])
        start_frame = int(row['start_frame'])
        end_frame = int(row['end_frame'])
        clip_id = str(row['clip_id'])
        clip = [clip for clip in clips if clip.clip_id==clip_id]
            # Initialize beamformer
        beamformer = Beamformer(
            session=session,
            inspection_id=inspection_id,
            clip=clip[0],
        )
    
        # Create subfolder for each bookmark
        bookmark_folder = output_path / f"bookmark_{idx}_track_{track_id:02d}"
        bookmark_folder.mkdir(exist_ok=True)
        
        print(f"Processing bookmark {idx}: track {track_id}, frames {start_frame}-{end_frame}")
        
        # Process frames in the range
        for frame in range(start_frame, end_frame + 1):
            try:
                # Beamform the frame
                vol = beamformer.beamform_data(
                    frame_range=(frame, frame + 1),
                    bf_type=BfImageType.InnerSurfaceDetect,
                    calc_circles=False,
                    bf_configs=FF_DEFAULTS,
                )
                
                # Create figure
                fig, ax = plt.subplots(figsize=(10, 8))
                
                # Display the beamformed image
                im = ax.imshow(vol[0], cmap='gray', aspect='auto')
                ax.set_title(f"Track {track_id}, Frame {frame}")
                ax.set_xlabel("Angle")
                ax.set_ylabel("Radial Distance")
                plt.colorbar(im, ax=ax, label="Intensity")
                
                # Save the figure
                output_file = bookmark_folder / f"frame_{frame:010d}.png"
                plt.savefig(output_file, dpi=150, bbox_inches='tight')
                plt.close(fig)
                
                print(f"  Saved frame {frame}")
                
            except Exception as e:
                print(f"  Error processing frame {frame}: {e}")
                continue
    
    print(f"\nAll frames saved to {output_folder}")


#visualize_bookmark_frames(bookmarks_dfs, session, output_folder="./dent_us_frames_visualization2")

In [10]:
def group_by_view_distance_bins(df, view_distance_start_col='view_distance_start', view_distance_stop_col='view_distance_stop', group_col='view_distance_group', bin_size=0.2):
    """
    Group DataFrame rows based on binned mean of view_distance_start and view_distance_stop
    
    Args:
        df: DataFrame with view distance columns
        view_distance_start_col: column name containing start view distances
        view_distance_stop_col: column name containing stop view distances
        group_col: name for the new grouping column
        bin_size: size of each bin (default 0.2)
    
    Returns:
        DataFrame with added grouping column
    """
    if df is None or len(df) == 0:
        return df
    
    df = df.copy()
    
    # Calculate mean of view_distance_start and view_distance_stop
    if view_distance_stop_col in df.columns:
        view_distance_mean = (df[view_distance_start_col] + df[view_distance_stop_col]) / 2
    else:
        # Fallback to just view_distance_start if view_distance_stop doesn't exist
        view_distance_mean = df[view_distance_start_col]
    
    # Create grouping column based on binned mean view distance
    df[group_col] = (np.floor(view_distance_mean / bin_size) * bin_size).round(1)
    
    return df

def analyze_view_distance_groups(df, view_distance_start_col='view_distance_start', view_distance_stop_col='view_distance_stop', group_col='view_distance_group', bin_size=0.2):
    """
    Group by view distance bins and analyze the groups
    
    Args:
        df: DataFrame with view distances
        view_distance_start_col: column name containing start view distances
        view_distance_stop_col: column name containing stop view distances
        group_col: name for the grouping column
        bin_size: size of each bin (default 0.2)
    
    Returns:
        tuple: (grouped_df, summary_stats)
    """
    # Add grouping column based on mean of start and stop
    grouped_df = group_by_view_distance_bins(df, view_distance_start_col, view_distance_stop_col, group_col, bin_size)
    
    # Create summary statistics with track_id list
    agg_dict = {
        view_distance_start_col: ['count', 'min', 'max', 'mean'],
        'track_id': lambda x: list(x.unique())  # Keep list of unique track IDs
    }
    
    # Add view_distance_stop aggregations if column exists
    if 'view_distance_stop' in df.columns:
        agg_dict['view_distance_stop'] = ['min', 'max', 'mean']
    
    # Add other columns if they exist
    #keep list of max_confidence values

    if 'avg_confidence' in df.columns:
        agg_dict['avg_confidence'] = ['mean', 'max', 'min']
    if 'sequence_length' in df.columns:
        agg_dict['sequence_length'] = ['mean', 'sum']
    
    # Perform the main aggregation without ransac_inlier_ratio_min
    summary_stats = grouped_df.groupby(group_col).agg(agg_dict)
    
    # Flatten column names
    new_columns = []
    for col in summary_stats.columns:
        if col[0] == 'track_id':
            new_columns.append('track_ids')
        else:
            new_columns.append(f'{col[0]}_{col[1]}')
    
    summary_stats.columns = new_columns
    summary_stats = summary_stats.reset_index()
    
    # Add ransac_inlier_ratio_min dictionary separately if column exists
    if 'ransac_inlier_ratio_min' in grouped_df.columns:
        ransac_dict = grouped_df.groupby(group_col).apply(
            lambda g: dict(zip(g['track_id'].values, g['ransac_inlier_ratio_min'].values))
        )
        summary_stats['ransac_inlier_ratio_min_by_track'] = summary_stats[group_col].map(ransac_dict)
    
    # Add max_confidence dictionary by track_id if column exists
    if 'max_confidence' in grouped_df.columns:
        max_conf_dict = grouped_df.groupby(group_col).apply(
            lambda g: dict(zip(g['track_id'].values, g['max_confidence'].values))
        )
        summary_stats['max_confidence_by_track'] = summary_stats[group_col].map(max_conf_dict)
    
    # Add count of unique tracks per group
    summary_stats['unique_tracks_count'] = summary_stats['track_ids'].apply(len)
    
    print(f"Created {len(summary_stats)} view distance groups (bin size: {bin_size})")
    print(f"Number of groups with less than 6 tracks: {len(summary_stats[summary_stats['unique_tracks_count'] <6])}")
    print(f"Group range: {summary_stats[group_col].min()} to {summary_stats[group_col].max()}")
    
    return grouped_df, summary_stats


# Apply to your bookmarks data with 0.2 bin size
bookmarks_dfs = bookmarks_dfs.query('frame_span <= 300 and frame_span >= 5')
grouped_bookmarks, group_summary = analyze_view_distance_groups(bookmarks_dfs, bin_size=0.3)

print("View Distance Groups Summary:")
display(group_summary)
display(grouped_bookmarks)


Created 13595 view distance groups (bin size: 0.3)
Number of groups with less than 6 tracks: 13261
Group range: 1.8 to 154593.0
View Distance Groups Summary:


,view_distance_group,view_distance_start_count,view_distance_start_min,view_distance_start_max,view_distance_start_mean,track_ids,view_distance_stop_min,view_distance_stop_max,view_distance_stop_mean,avg_confidence_mean,avg_confidence_max,avg_confidence_min,sequence_length_mean,sequence_length_sum,max_confidence_by_track,unique_tracks_count
0,1.8,1,1.846727,1.846727,1.846727,[5],2.158097,2.158097,2.158097,0.994900,0.9949,0.9949,208.0,208,{5: 0.9963},1
1,2.4,4,2.503118,2.620092,2.564977,"[1, 4]",2.550965,2.647089,2.597936,0.990875,0.9920,0.9903,23.0,92,"{1: 0.9905, 4: 0.9932}",2
2,2.7,2,2.696583,2.719080,2.707831,[1],2.714580,2.894558,2.804569,0.991250,0.9921,0.9904,65.5,131,{1: 0.9935},1
3,3.0,2,2.975548,3.142027,3.058787,[1],3.134527,3.298007,3.216267,0.991650,0.9918,0.9915,106.0,212,{1: 0.9924},1
4,3.3,2,3.368498,3.557474,3.462986,[1],3.411992,3.564973,3.488483,0.990650,0.9910,0.9903,18.0,36,{1: 0.9904},1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13590,154590.0,6,154589.960133,154590.105011,154590.038148,"[2, 3, 5, 6, 8, 16]",154590.048638,154590.178794,154590.137013,0.991283,0.9932,0.9903,66.666667,400,"{2: 0.9905, 3: 0.993, 5: 0.9912, 6: 0.9908, 8:...",6
13591,154591.2,6,154591.324890,154591.480778,154591.399876,"[13, 15, 17, 18]",154591.376393,154591.493903,154591.444702,0.992233,0.9941,0.9904,30.0,180,"{13: 0.996, 15: 0.9946, 17: 0.9906, 18: 0.9923}",4
13592,154591.5,7,154591.319977,154591.708169,154591.571648,"[11, 12, 13, 17, 18, 19]",154591.601084,154591.748164,154591.687169,0.993186,0.9950,0.9906,77.142857,540,"{11: 0.996, 12: 0.9966, 13: 0.9933, 17: 0.9927...",6
13593,154591.8,4,154591.759078,154591.866106,154591.831231,"[12, 13, 16, 18]",154591.858073,154591.872450,154591.865374,0.992100,0.9950,0.9903,23.75,95,"{12: 0.9917, 13: 0.9903, 16: 0.9928, 18: 0.9964}",4


,inspection_id,frame_range,clip_id,track_id,start_frame,end_frame,sequence_length,frame_span,num_detections,start_timer_tick,...,start_odometer_tick,end_odometer_tick,odometer_span,avg_confidence,max_confidence,min_confidence,confidence_std,view_distance_start,view_distance_stop,view_distance_group
0,0AS0KUQOE45,100105009_100105013,01-000-198EUYDF,0,100105009,100105013,5,5,5,1863376033,...,1509379286,1509379346,60,0.9903,0.9903,0.9903,0.0000,150934.720981,150934.727119,150934.5
1,0AS0KUQOE45,100138954_100139033,01-000-198EUYDF,0,100138954,100139033,80,80,80,1863988282,...,1509890571,1509891756,1185,0.9927,0.9934,0.9910,0.0007,150985.849473,150985.968074,150985.8
2,0AS0KUQOE45,100139064_100139153,01-000-198EUYDF,0,100139064,100139153,90,90,90,1863990191,...,1509892221,1509893556,1335,0.9908,0.9920,0.9900,0.0006,150986.014634,150986.148134,150985.8
3,0AS0KUQOE45,100139189_100139283,01-000-198EUYDF,0,100139189,100139283,95,95,95,1863992393,...,1509894096,1509895506,1410,0.9918,0.9936,0.9901,0.0012,150986.202111,150986.343168,150986.1
4,0AS0KUQOE45,100141729_100141733,01-000-198EUYDF,0,100141729,100141733,5,5,5,1864036084,...,1509932211,1509932271,60,0.9903,0.9903,0.9903,0.0000,150990.013773,150990.019792,150990.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30092,0AS0KUQOE45,99951679_99951683,01-021-198EVB8A,21,99951679,99951683,5,5,5,1860809757,...,1507212516,1507212576,60,0.9902,0.9902,0.9902,0.0000,150717.076736,150717.082713,150717.0
30093,0AS0KUQOE45,99958859_99959018,01-021-198EVB8A,21,99958859,99959018,160,160,160,1860936884,...,1507321406,1507323836,2430,0.9928,0.9942,0.9906,0.0009,150727.965643,150728.208832,150727.8
30094,0AS0KUQOE45,99959154_99959268,01-021-198EVB8A,21,99959154,99959268,115,115,115,1860942135,...,1507325906,1507327646,1740,0.9920,0.9925,0.9904,0.0005,150728.415764,150728.589824,150728.4
30095,0AS0KUQOE45,99961669_99961673,01-021-198EVB8A,21,99961669,99961673,5,5,5,1860986503,...,1507363921,1507363981,60,0.9901,0.9901,0.9901,0.0000,150732.217364,150732.223350,150732.0


In [11]:
group_summary = group_summary[group_summary['unique_tracks_count']<6]
len(group_summary)

13261

In [12]:
def combine_consecutive_rows_with_equal_tracks(df, track_ids_col='track_ids', 
                                               agg_functions=None,
                                               view_distance_stop_col='view_distance_stop_mean',
                                               view_distance_start_col='view_distance_start_mean',
                                               max_gap=1.0):
    """
    Combine consecutive rows in a DataFrame based on track_ids list comparison:
    - If lists are entirely equal (same track_ids): merge them
    - If lengths are not equal:
      - If either list has length 2: merge if they have 1 common track_id
      - If both lists have length > 2: merge if they have at least 2 common track_ids
    
    Additional condition: Do not merge if view_distance_start_max from first row 
    is more than max_gap smaller than view_distance_start_min in second row.
    
    Args:
        df: DataFrame to process
        track_ids_col: column name containing track_ids list
        agg_functions: dict mapping column names to aggregation functions
                      (e.g., {'view_distance_start_mean': 'mean', 'count': 'sum'})
                      If None, uses default aggregations
        view_distance_stop_col: column name for view distance stop (from first row)
        view_distance_start_col: column name for view distance start (from second row)
        max_gap: maximum allowed gap between rows (default: 1.0)
    
    Returns:
        DataFrame with combined rows
    """
    if df is None or len(df) == 0:
        return df
    
    df = df.copy().reset_index(drop=True)
    
    # Default aggregation functions
    if agg_functions is None:
        agg_functions = {}
    
    # Helper function to check if a value is numeric-compatible
    def is_numeric_type(val):
        """Check if value can be used in numeric operations"""
        return isinstance(val, (int, float, np.integer, np.floating)) and not isinstance(val, bool)
    
    # Helper function to check if column contains non-numeric types
    def column_has_numeric_values(series):
        """Check if series contains numeric values"""
        if len(series) == 0:
            return False
        first_val = series.iloc[0]
        # Check if first value is numeric
        if pd.isna(first_val):
            # Check other values
            for val in series:
                if not pd.isna(val):
                    return is_numeric_type(val)
            return False
        return is_numeric_type(first_val)
    
    # Helper function to merge dictionaries (for max_confidence_by_track, etc.)
    def merge_dicts_max(dict_list):
        """
        Merge multiple dictionaries, taking maximum value for each key.
        Used for max_confidence_by_track and similar dictionary columns.
        """
        if not dict_list:
            return {}
        
        # Filter out None/NaN values
        valid_dicts = [d for d in dict_list if d is not None and isinstance(d, dict)]
        
        if not valid_dicts:
            return {}
        
        # Start with first dictionary
        merged = valid_dicts[0].copy()
        
        # For each subsequent dictionary, take max value for each key
        for d in valid_dicts[1:]:
            for key, value in d.items():
                if key in merged:
                    # Take maximum value
                    merged[key] = max(merged[key], value)
                else:
                    # Add new key
                    merged[key] = value
        
        return merged
    
    # Helper function to check if two track_ids lists should be merged
    def tracks_should_merge(tracks1, tracks2):
        """
        Check if two track_ids lists should be merged based on:
        - If lists are entirely equal (same track_ids): merge them
        - If lengths are not equal:
          - If either list has length 2: merge if they have 1 common track_id
          - If both lists have length > 2: merge if they have at least 2 common track_ids
        """
        # Handle None/NaN cases
        if tracks1 is None and tracks2 is None:
            return True
        if tracks1 is None or tracks2 is None:
            return False
        
        # Check if either is NaN (for scalar values)
        try:
            if pd.isna(tracks1) and pd.isna(tracks2):
                return True
            if pd.isna(tracks1) or pd.isna(tracks2):
                return False
        except (ValueError, TypeError):
            # If pd.isna fails (e.g., for lists), continue with comparison
            pass
        
        # Convert to lists if needed
        if isinstance(tracks1, (list, tuple, np.ndarray)):
            list1 = list(tracks1)
        else:
            list1 = [tracks1] if tracks1 is not None else []
        
        if isinstance(tracks2, (list, tuple, np.ndarray)):
            list2 = list(tracks2)
        else:
            list2 = [tracks2] if tracks2 is not None else []
        
        # Get lengths
        len1 = len(list1)
        len2 = len(list2)
        
        # Handle empty lists
        if len1 == 0 and len2 == 0:
            return True
        if len1 == 0 or len2 == 0:
            return False
        
        # Convert to sets for comparison
        set1 = set(list1)
        set2 = set(list2)
        common_tracks = set1.intersection(set2)
        num_common = len(common_tracks)
        
        # If lists are entirely equal (same track_ids): merge them
        if set1 == set2:
            return True
        
        # If lengths are not equal:
        # - If either list has length 2: merge if they have 1 common track_id
        # - If both lists have length > 2: merge if they have at least 2 common track_ids
        if len1 == 2 or len2 == 2:
            # At least one list has length 2: merge if 1 common track_id
            return num_common >= 1
        else:
            # Both lists have length > 2: merge if at least 2 common track_ids
            return num_common >= 2
    
    def view_distance_gap_ok(row1, row2):
        """Check if view distance gap is acceptable for merging."""
        if view_distance_stop_col not in df.columns or view_distance_start_col not in df.columns:
            return True
        
        vd_stop_1 = row1[view_distance_stop_col]
        vd_start_2 = row2[view_distance_start_col]
        
        if pd.isna(vd_stop_1) or pd.isna(vd_start_2):
            return True
        
        if vd_stop_1 + max_gap < vd_start_2:
            return False
        
        return True
    
    # Identify groups of consecutive rows with common track_ids
    groups = []
    current_group = [0]
    
    for i in range(1, len(df)):
        prev_tracks = df.loc[i-1, track_ids_col]
        curr_tracks = df.loc[i, track_ids_col]
        prev_row = df.loc[i-1]
        curr_row = df.loc[i]
        
        # Check both conditions: track_ids and view distance gap
        tracks_ok = tracks_should_merge(prev_tracks, curr_tracks)
        view_distance_ok = view_distance_gap_ok(prev_row, curr_row)
        
        if tracks_ok and view_distance_ok:
            # Both conditions satisfied - add to current group
            current_group.append(i)
        else:
            # Don't meet merge criteria - save current group and start new one
            groups.append(current_group)
            current_group = [i]
    
    # Add the last group
    groups.append(current_group)
    
    # Combine rows in each group
    combined_rows = []
    
    for group_indices in groups:
        group_df = df.loc[group_indices].copy()
        
        # Create combined row
        combined_row = {}
        
        for col in df.columns:
            if col == track_ids_col:
                # Combine track_ids: union of all track_ids in the group
                all_tracks = []
                for idx in group_indices:
                    tracks = df.loc[idx, track_ids_col]
                    if tracks is not None:
                        if isinstance(tracks, (list, tuple, np.ndarray)):
                            all_tracks.extend(list(tracks))
                        else:
                            all_tracks.append(tracks)
                # Remove duplicates and sort
                combined_row[col] = sorted(list(set(all_tracks)))
            
            elif col in agg_functions:
                # Use specified aggregation function
                agg_func = agg_functions[col]
                
                # Check if column contains non-numeric types (dicts, lists, etc.)
                if not column_has_numeric_values(group_df[col]):
                    # Special handling for dictionary columns
                    first_val = group_df[col].iloc[0]
                    if isinstance(first_val, dict):
                        # Merge dictionaries - collect all dicts and merge them
                        dict_list = group_df[col].tolist()
                        if col in ['max_confidence_by_track', 'ransac_inlier_ratio_min_by_track']:
                            # For max_confidence: take max value for each track_id
                            # For ransac_inlier_ratio_min: take min value for each track_id
                            if col == 'max_confidence_by_track':
                                combined_row[col] = merge_dicts_max(dict_list)
                            elif col == 'ransac_inlier_ratio_min_by_track':
                                # For min ratio, we want minimum values
                                merged = {}
                                valid_dicts = [d for d in dict_list if d is not None and isinstance(d, dict)]
                                if valid_dicts:
                                    merged = valid_dicts[0].copy()
                                    for d in valid_dicts[1:]:
                                        for key, value in d.items():
                                            if key in merged:
                                                merged[key] = min(merged[key], value)
                                            else:
                                                merged[key] = value
                                combined_row[col] = merged
                            else:
                                # Default: merge and take max
                                combined_row[col] = merge_dicts_max(dict_list)
                        else:
                            # For other dict columns, merge with max
                            combined_row[col] = merge_dicts_max(dict_list)
                    else:
                        # For non-numeric, non-dict types, use 'first'
                        combined_row[col] = group_df[col].iloc[0]
                elif agg_func == 'first':
                    combined_row[col] = group_df[col].iloc[0]
                elif agg_func == 'last':
                    combined_row[col] = group_df[col].iloc[-1]
                elif agg_func == 'mean':
                    combined_row[col] = group_df[col].mean()
                elif agg_func == 'sum':
                    combined_row[col] = group_df[col].sum()
                elif agg_func == 'min':
                    combined_row[col] = group_df[col].min()
                elif agg_func == 'max':
                    combined_row[col] = group_df[col].max()
                elif agg_func == 'list':
                    # Combine into list
                    combined_row[col] = group_df[col].tolist()
                else:
                    # Default to first value
                    combined_row[col] = group_df[col].iloc[0]
            
            else:
                # Default aggregation based on column type
                if column_has_numeric_values(group_df[col]):
                    # Numeric: take mean
                    combined_row[col] = group_df[col].mean()
                else:
                    # Non-numeric: check if it's a dictionary
                    first_val = group_df[col].iloc[0]
                    if isinstance(first_val, dict):
                        # Merge dictionaries - collect all dicts and merge them
                        dict_list = group_df[col].tolist()
                        if col == 'max_confidence_by_track':
                            # For max_confidence: take max value for each track_id
                            combined_row[col] = merge_dicts_max(dict_list)
                        elif col == 'ransac_inlier_ratio_min_by_track':
                            # For min ratio: take min value for each track_id
                            merged = {}
                            valid_dicts = [d for d in dict_list if d is not None and isinstance(d, dict)]
                            if valid_dicts:
                                merged = valid_dicts[0].copy()
                                for d in valid_dicts[1:]:
                                    for key, value in d.items():
                                        if key in merged:
                                            merged[key] = min(merged[key], value)
                                        else:
                                            merged[key] = value
                            combined_row[col] = merged
                        else:
                            # Default: merge with max
                            combined_row[col] = merge_dicts_max(dict_list)
                    else:
                        # Non-numeric (lists, strings): take first value
                        combined_row[col] = group_df[col].iloc[0]
        
        combined_rows.append(combined_row)
    
    result_df = pd.DataFrame(combined_rows)
    
    print(f"Combined {len(df)} rows into {len(result_df)} rows")
    
    return result_df

In [13]:
# custom aggregation functions
agg_funcs = {
    'view_distance_start_min': 'min',
    'view_distance_stop_max': 'max',
    'view_distance_start_mean': 'min',
    'view_distance_stop_mean': 'max',
    'view_distance_start_count': 'sum',
    'unique_tracks_count': 'first',  # Keep first value since track_ids are same
    'view_distance_group': 'first',
    'max_confidence_by_track': 'sum',  # Keep first dict
    'ransac_inlier_ratio_min_by_track': 'first'
}

combined_df = combine_consecutive_rows_with_equal_tracks(
    group_summary, 
    track_ids_col='track_ids',
    agg_functions=agg_funcs,
    max_gap=0.3
)


Combined 13261 rows into 8589 rows


In [14]:
a_old = group_summary["view_distance_stop_mean"]-group_summary["view_distance_start_mean"]
a = combined_df["view_distance_stop_mean"]-combined_df["view_distance_start_mean"]
b = combined_df["view_distance_stop_max"]-combined_df["view_distance_start_max"]

In [15]:
#give the top 5 indicies of a with max values
top_5_indices = a.nlargest(5
                           ).index.tolist()
print("Top 5 indices with max (view_distance_stop_mean - view_distance_start_mean):", top_5_indices)
#print combined rows at these indicies
print(a.loc[top_5_indices])
print("Combined rows at top 5 indices:")
display(combined_df.loc[top_5_indices])

Top 5 indices with max (view_distance_stop_mean - view_distance_start_mean): [6190, 5779, 1039, 3388, 7364]
6190    5.767368
5779    4.653304
1039    4.094543
3388    3.743371
7364    3.704281
dtype: float64
Combined rows at top 5 indices:


,view_distance_group,view_distance_start_count,view_distance_start_min,view_distance_start_max,view_distance_start_mean,track_ids,view_distance_stop_min,view_distance_stop_max,view_distance_stop_mean,avg_confidence_mean,avg_confidence_max,avg_confidence_min,sequence_length_mean,sequence_length_sum,max_confidence_by_track,unique_tracks_count
6190,129532.5,82,129532.559298,129535.583811,129532.652117,"[1, 4, 6, 10, 11, 12, 14, 15, 16, 17, 20]",129535.471650,129538.620332,129538.419485,0.991199,0.992200,0.990405,82.691667,317.500000,"{14: 0.9928, 16: 0.9936, 20: 0.9936, 15: 0.994...",3
5779,125493.9,18,125493.849069,125496.217075,125493.849069,[18],125496.360882,125498.502373,125498.502373,0.991728,0.991781,0.991675,106.406250,110.312500,{18: 0.9955},1
1039,23806.5,59,23806.706609,23808.815439,23806.706609,"[0, 1, 10, 11, 12, 13, 21]",23808.690754,23810.955161,23810.801152,0.990809,0.991400,0.990387,46.375000,174.333333,"{12: 0.9948, 0: 0.9932, 13: 0.9932, 21: 0.9933...",1
3388,85548.3,22,85548.307062,85550.218367,85548.379571,[9],85550.262435,85552.122942,85552.122942,0.991841,0.992169,0.991515,90.641026,125.000000,{9: 0.9947},1
7364,144725.1,76,144725.107587,144727.118333,144725.204843,"[2, 3, 4, 6, 7, 10, 11, 13, 14]",144726.980036,144729.033119,144728.909124,0.991682,0.993208,0.990531,60.362887,336.538462,"{2: 0.9936, 7: 0.9968, 13: 0.9933, 14: 0.9933,...",4


In [16]:
def add_final_track_id(df, track_ids_col='track_ids', output_col='final_track_id'):
    """
    Add a final_track_id column that contains the track_id from the list 
    that is closest to all other track_ids (minimizes sum of distances).
    
    Args:
        df: DataFrame with track_ids column
        track_ids_col: column name containing list of track_ids
        output_col: name for the new final_track_id column
    
    Returns:
        DataFrame with added final_track_id column
    """
    if df is None or len(df) == 0:
        return df
    
    df = df.copy()
    final_track_ids = []
    
    def find_closest_track(track_list):
        """
        Find the track_id that minimizes the sum of absolute differences 
        to all other track_ids in the list.
        """
        if track_list is None or len(track_list) == 0:
            return None
        
        # Convert to list if needed
        if isinstance(track_list, (list, tuple, np.ndarray)):
            tracks = list(track_list)
        else:
            tracks = [track_list]
        
        # Remove any None/NaN values
        tracks = [t for t in tracks if t is not None and not pd.isna(t)]
        
        if len(tracks) == 0:
            return None
        
        # If only one track, return it
        if len(tracks) == 1:
            return tracks[0]
        
        # Convert to numeric (in case they're strings)
        try:
            tracks = [int(t) for t in tracks]
        except (ValueError, TypeError):
            # If conversion fails, use as-is
            pass
        
        # Find track_id that minimizes sum of absolute differences to all others
        min_total_distance = float('inf')
        closest_track = tracks[0]
        
        for candidate_track in tracks:
            # Calculate sum of absolute differences to all other tracks
            total_distance = sum(abs(candidate_track - other_track) for other_track in tracks)
            
            if total_distance < min_total_distance:
                min_total_distance = total_distance
                closest_track = candidate_track
        
        return closest_track
    
    # Process each row
    for idx, row in df.iterrows():
        track_list = row[track_ids_col]
        final_track = find_closest_track(track_list)
        final_track_ids.append(final_track)
    
    df[output_col] = final_track_ids
    
    return df

In [19]:
df_with_final_track = add_final_track_id(combined_df, track_ids_col='track_ids', output_col='final_track_id')

In [20]:
# Convert dict keys to strings for parquet compatibility
def convert_dict_keys_to_str(d):
    """Convert dictionary keys to strings for parquet compatibility."""
    if d is None or not isinstance(d, dict):
        return d
    return {str(k): v for k, v in d.items()}

# Apply to columns with dict values
df_to_save = df_with_final_track.copy()
if 'max_confidence_by_track' in df_to_save.columns:
    df_to_save['max_confidence_by_track'] = df_to_save['max_confidence_by_track'].apply(convert_dict_keys_to_str)
# Save to parquet
df_to_save.to_parquet(f"./insp_{inspection_id}_final_df_us_with_track.parquet", index=False)


## Upload anomalies

In [ ]:

display(df_with_final_track.iloc[111


])


view_distance_group                                                       33.0
view_distance_start_count                                                   17
view_distance_start_min                                              32.981216
view_distance_start_max                                              33.548367
view_distance_start_mean                                              33.11676
track_ids                                         [0, 2, 3, 5, 11, 14, 15, 18]
view_distance_stop_min                                               33.403471
view_distance_stop_max                                               33.984473
view_distance_stop_mean                                              33.832651
avg_confidence_mean                                                   0.991106
avg_confidence_max                                                      0.9923
avg_confidence_min                                                      0.9903
sequence_length_mean                                

In [24]:
from ilipy import ClipTypes, Session, TrackIndex, ViewDistance
from ilipy.channeldata import ImageProfile
from ilipy.features import Bookmarks
from ilipyutils.ml_features.base import (get_ml_models_info_list, AnomalyStatus)
from ilipyutils.ml_features.insert import FeatureInsert, GeometryCubeParameters
from ili_custom_data.load_model import ModelDataLoader
from ilipy.database import DistanceCorrelation
from ilipyutils.ml_features.overlap import (
    AnomalyOverlapManager,
    OverlapAction,
    OverlapPolicy,
)


session = Session(environment=environment)
session.set_active_inspection(inspection_id)
dist_corr = DistanceCorrelation(session)
bookmarks_interface = Bookmarks(session=session)
latest_model = ModelDataLoader.get_latest_model()

overlap_policy = OverlapPolicy(
    action=OverlapAction.SKIP,
    min_overlap_iou=0.1,
)
anomaly_overlap_manager = AnomalyOverlapManager(
    overlap_policy=overlap_policy,
    session=session,
    bookmarks_interface=bookmarks_interface,
)
feature_insert = FeatureInsert(
    session=session,
    bookmarks_interface=bookmarks_interface,
    #anomaly_overlap_manager=anomaly_overlap_manager, # Set to None to disable overlap checking
)

PyAuthenticator.cpp(239): Attempting an IAM role login as no username and password or environment variables were provided.

CognitoAuthenticator.cpp(158): Using Cognito Credentials

CognitoAuthenticator.cpp(842): Login Successful.

CognitoAuthenticator.cpp(883): Logged in as username: zahra.mirikharaji

RequestPool.cpp(84): [HttpRequestPool] Host: dvv-storage-01.darkvision.local, Port: 8070, Concurrent Connections: 1

RequestPool.cpp(84): [HttpRequestPool] Host: storage-dvh-01.darkvision.local, Port: 8070, Concurrent Connections: 1

RequestPool.cpp(84): [HttpRequestPool] Host: dvv-storage-02.darkvision.local, Port: 8070, Concurrent Connections: 1

RestApi.cpp(84): ApiService Ping Response (http://dvv-storage-02.darkvision.local:8070) : 200, 10ms

RestApi.cpp(84): ApiService Ping Response (http://dvv-storage-01.darkvision.local:8070) : 200, 27ms

RequestPool.cpp(303): Failed to acquire connection: http://storage-dvh-01.darkvision.local:8070/ping GET code: 1048, value: AWS_IO_SOCKET_TIME

In [25]:
import numpy as np
cube_params = GeometryCubeParameters(
tlbr_cube_angle_rad=(np.deg2rad(-15), np.deg2rad(15)),
tlbr_probe_scan_angle_rad=(np.deg2rad(-15), np.deg2rad(15)),
tlbr_radial_position_mm=(200-(1.9/2), 200+(1.9/2)),
)
model_info = next(
    model
    for model in get_ml_models_info_list()
    if model.model_name == "Ultrasound-Dent-v1"
)
model_info


MLModelInfo(model_name='Ultrasound-Dent-v1', model_id=14, model_score_bound=[0, 1], anomaly_type_name='DentPlain')

In [26]:

#reset index of final_grouped_bookmarks
final_grouped_bookmarks = df_with_final_track.reset_index(drop=True)
display(final_grouped_bookmarks)

,view_distance_group,view_distance_start_count,view_distance_start_min,view_distance_start_max,view_distance_start_mean,track_ids,view_distance_stop_min,view_distance_stop_max,view_distance_stop_mean,avg_confidence_mean,avg_confidence_max,avg_confidence_min,sequence_length_mean,sequence_length_sum,max_confidence_by_track,unique_tracks_count,final_track_id
0,1.8,1,1.846727,1.846727,1.846727,[5],2.158097,2.158097,2.158097,0.994900,0.994900,0.994900,208.000000,208.00,{5: 0.9963},1,5
1,2.4,10,2.503118,3.009668,2.564977,"[1, 4]",2.953016,3.564973,3.488483,0.991106,0.991725,0.990625,53.125000,117.75,"{1: 0.9935, 4: 0.9932}",2,1
2,4.2,1,4.206906,4.206906,4.206906,[4],4.295079,4.295079,4.295079,0.992300,0.992300,0.992300,60.000000,60.00,{4: 0.9932},1,4
3,4.8,4,4.786692,5.043853,4.913261,"[2, 13, 21]",4.851275,5.064899,4.966871,0.991375,0.994200,0.990300,37.500000,150.00,"{2: 0.9958, 13: 0.9905, 21: 0.9909}",3,13
4,5.7,5,5.671522,5.933353,5.803508,"[2, 4, 5, 6]",5.743706,5.961843,5.868238,0.993060,0.994000,0.991600,45.000000,225.00,"{2: 0.9924, 4: 0.9943, 5: 0.9952, 6: 0.9951}",4,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8584,154587.3,2,154587.489996,154587.573628,154587.531812,"[6, 7]",154587.572590,154587.594751,154587.583670,0.991400,0.991600,0.991200,35.000000,70.00,"{6: 0.9918, 7: 0.9919}",2,6
8585,154589.7,7,154589.727087,154589.989450,154589.854374,"[2, 3, 4, 13, 18]",154589.837648,154590.079200,154589.977331,0.991314,0.993100,0.990200,82.857143,580.00,"{2: 0.994, 3: 0.9905, 4: 0.9915, 13: 0.9931, 1...",5,4
8586,154591.2,6,154591.324890,154591.480778,154591.399876,"[13, 15, 17, 18]",154591.376393,154591.493903,154591.444702,0.992233,0.994100,0.990400,30.000000,180.00,"{13: 0.996, 15: 0.9946, 17: 0.9906, 18: 0.9923}",4,15
8587,154591.8,4,154591.759078,154591.866106,154591.831231,"[12, 13, 16, 18]",154591.858073,154591.872450,154591.865374,0.992100,0.995000,0.990300,23.750000,95.00,"{12: 0.9917, 13: 0.9903, 16: 0.9928, 18: 0.9964}",4,13


In [ ]:
# anomalies = bookmarks_interface.get_anomalies(inspectionId=inspection_id)
# dent_anomalies = [a for a in anomalies if "Ultrasound-Dent-v1" in a.tags]
# print(f"{len(dent_anomalies)} anomalies with Ultrasound-Dent-v1 tag found in {environment}.")


In [ ]:
# 
# deleted_anomalies = [bookmarks_interface.delete_anomaly(a.feature.anomaly_feature_id) for a in dent_anomalies]

In [37]:

for i, row in final_grouped_bookmarks.iterrows():
    if i in list(range(0,112)):
        continue


    
       
    profile = ImageProfile.ZeroAngle 
    vd_start, vd_end, track_id = row['view_distance_start_mean'], row['view_distance_stop_mean'], row['final_track_id']

        
    track_nums = (track_id,track_id)
    start_frame_odo_ticks = dist_corr.get_odometer_ticks_from_view_distance(TrackIndex(track_nums[0]), ViewDistance(vd_start)).value
    end_frame_odo_ticks = dist_corr.get_odometer_ticks_from_view_distance(TrackIndex(track_nums[1]), ViewDistance(vd_end)).value

    print(f"uploading row {i+1} of {len(final_grouped_bookmarks)}")
    anomaly_info = feature_insert.insert_ml_pred(
    inspection_id=inspection_id,
    tlbr_track_indices=track_nums,
    tlbr_odometer_ticks=(start_frame_odo_ticks, end_frame_odo_ticks),
    cube_params=cube_params,
    model_info=model_info,
    anomaly_status=AnomalyStatus.REVIEW_DETECTION,
    image_profile=profile,
    extra_tags=None,
    # ili_custom_data=model_instance
    )



uploading row 113 of 8589
uploading row 114 of 8589
uploading row 115 of 8589
uploading row 116 of 8589
uploading row 117 of 8589
uploading row 118 of 8589
uploading row 119 of 8589
uploading row 120 of 8589
uploading row 121 of 8589
uploading row 122 of 8589
uploading row 123 of 8589
uploading row 124 of 8589
uploading row 125 of 8589
uploading row 126 of 8589
uploading row 127 of 8589
uploading row 128 of 8589
uploading row 129 of 8589
uploading row 130 of 8589
uploading row 131 of 8589
uploading row 132 of 8589
uploading row 133 of 8589
uploading row 134 of 8589
uploading row 135 of 8589
uploading row 136 of 8589
uploading row 137 of 8589
uploading row 138 of 8589
uploading row 139 of 8589
uploading row 140 of 8589
uploading row 141 of 8589
uploading row 142 of 8589
uploading row 143 of 8589
uploading row 144 of 8589
uploading row 145 of 8589
uploading row 146 of 8589
uploading row 147 of 8589
uploading row 148 of 8589
uploading row 149 of 8589
uploading row 150 of 8589
uploading ro

CognitoAuthenticator.cpp(919): Error refreshing Tokens: Refresh Token has expired

CognitoAuthenticator.cpp(752): Global Signout Successful...

